# AOI Dispersion Workshop — Solved Version

This solved copy keeps the full teaching notes from the student version, but the exercise cells contain one reference implementation.

This workshop analyses the accuracy and precision of gaze relative to the
IMRF experiment's stimulus areas of interest (AOIs). It reads the
`apriltag_mapped_fixations.csv` produced by the surface mapping workshop
(notebook 02) and walks you through the core dispersion metrics used in
eye-tracking research.

> **Prerequisite:** Run `02_surface_mapping_workshop_student.ipynb` (or the
> solved version) through the export cell before opening this notebook.


## Learning Goals

By the end, you should be able to:

1. Compute the **centroid bias** — how far, on average, gaze lands from the target.
2. Compute the **RMSE** — a joint accuracy metric that penalises individual outliers.
3. Count how many fixations fall **inside** a circular AOI.
4. Convert pixel distances to **degrees of visual angle** using the full arctan formula.
5. Compute the **BCEA** — a standard eye-tracking precision metric based on the bivariate scatter ellipse.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon as MplPolygon
from IPython.display import display

from nb_setup import DEFAULT_PARTICIPANT_ID, setup
PROJECT_ROOT = setup("aoi.py")

from libs.analysis.aoi import (
    AOI_COLORS,
    build_paradigm_aois,
    classify_fixations_extended,
    compute_voronoi_regions,
    compute_grid_cells,
    pixels_per_degree,
)
from libs.analysis.fixation_utils import confidence_ellipse_params, bcea
from libs.analysis.recording_helpers import find_neon_recording, neon_participant_id
from libs.project_config import PLOT_COLORS

Project root: /Users/eduardo/Workspace/Unity/UnityProjects/IMRF_2026_SpatialAudio_Eyetracking/IMRF_Eyetracking


## Exercise 8 — Euclidean Bias

**Accuracy** measures how close gaze lands to the intended target *on average*.
The simplest accuracy metric is the **centroid bias**: compute the mean gaze
position (the centroid), then measure how far that centroid is from the target
center.

Because the offset has both an x and a y component, the scalar distance is the
Euclidean (straight-line) norm:

$$\text{bias} = \|\bar{\mathbf{g}} - \mathbf{c}\|_2 = \sqrt{(\bar{x} - c_x)^2 + (\bar{y} - c_y)^2}$$

`np.hypot(a, b)` computes √(a² + b²) correctly even for very small values.


In [2]:
def euclidean_bias(
    xs: np.ndarray,
    ys: np.ndarray,
    cx: float,
    cy: float,
) -> float:
    """Return the Euclidean distance from the fixation centroid to (cx, cy)."""
    if len(xs) == 0:
        return 0.0
    return float(np.hypot(float(xs.mean()) - cx, float(ys.mean()) - cy))

In [3]:
def test_euclidean_bias(fn):
    xs = np.array([3.0, 3.0, 3.0])
    ys = np.array([4.0, 4.0, 4.0])
    assert np.isclose(fn(xs, ys, 0.0, 0.0), 5.0)

    xs2 = np.array([10.0, 12.0])
    ys2 = np.array([20.0, 20.0])
    assert np.isclose(fn(xs2, ys2, 11.0, 20.0), 0.0)

    assert fn(np.array([]), np.array([]), 0.0, 0.0) == 0.0

    print("Exercise 8 passed.")


test_euclidean_bias(euclidean_bias)

Exercise 9 passed.


## Pre-solved — RMSE from Target Centre

The centroid bias measures offset of the *mean* position. It can mask a
systematic bias in one axis that happens to cancel across fixations. A more
complete accuracy metric is the **Root Mean Square Error (RMSE)**: average
the squared distance of every individual fixation from the target, then take
the square root:

$$\text{RMSE} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} \left[(x_i - c_x)^2 + (y_i - c_y)^2\right]}$$

RMSE is always ≥ bias; the gap between them reflects spread around the centroid.


In [4]:
def rmse_from_center(
    xs: np.ndarray,
    ys: np.ndarray,
    cx: float,
    cy: float,
) -> float:
    """Return the RMSE of fixations from (cx, cy)."""
    if len(xs) == 0:
        return 0.0
    dists_sq = (xs - cx) ** 2 + (ys - cy) ** 2
    return float(np.sqrt(np.mean(dists_sq)))

In [5]:
def test_rmse_from_center(fn):
    xs = np.array([3.0, -3.0])
    ys = np.array([4.0, -4.0])
    assert np.isclose(fn(xs, ys, 0.0, 0.0), 5.0)

    xs0 = np.array([10.0, 10.0, 10.0])
    ys0 = np.array([20.0, 20.0, 20.0])
    assert np.isclose(fn(xs0, ys0, 10.0, 20.0), 0.0)

    xm = np.array([0.0, 6.0])
    ym = np.array([0.0, 0.0])
    assert fn(xm, ym, 0.0, 0.0) >= euclidean_bias(xm, ym, 0.0, 0.0) - 1e-9
    assert fn(np.array([]), np.array([]), 0.0, 0.0) == 0.0

    print("Pre-solved RMSE helper passed.")


test_rmse_from_center(rmse_from_center)

Pre-solved RMSE helper passed.


## Pre-solved — Count Inside a Circular AOI

An AOI "hit" means a fixation landed inside the circular region centred on
the target. The check is simple: compute the Euclidean distance from each
fixation to the AOI centre and compare it to the AOI radius.

Complete `count_inside_aoi`. Return both the count and the fraction
(proportion of all fixations that hit the AOI).


In [6]:
def count_inside_aoi(
    xs: np.ndarray,
    ys: np.ndarray,
    cx: float,
    cy: float,
    radius: float,
) -> tuple[int, float]:
    """Count and fraction of fixations within a circular AOI."""
    if len(xs) == 0:
        return (0, 0.0)
    dists = np.hypot(xs - cx, ys - cy)
    n_inside = int(np.sum(dists <= radius))
    return (n_inside, n_inside / len(xs))

In [7]:
def test_count_inside_aoi(fn):
    xs = np.array([0.0, 1.0, 10.0, 0.0])
    ys = np.array([0.0, 0.0,  0.0, 10.0])
    n, frac = fn(xs, ys, 0.0, 0.0, 5.0)
    assert isinstance(n, int) and n == 2
    assert np.isclose(frac, 0.5)

    n_b, _ = fn(np.array([5.0]), np.array([0.0]), 0.0, 0.0, 5.0)
    assert n_b == 1, "Boundary point must count as a hit."

    n_e, f_e = fn(np.array([]), np.array([]), 0.0, 0.0, 5.0)
    assert n_e == 0 and f_e == 0.0

    print("Pre-solved AOI-count helper passed.")


test_count_inside_aoi(count_inside_aoi)

Pre-solved AOI-count helper passed.


## Exercise 9 — Pixels per Degree of Visual Angle

All the metrics above are in pixels. To compare across studies, setups, or
participants we convert to **degrees of visual angle (°)** — the angle
subtended at the eye by a given physical extent.

For this workshop we use the local 1-degree conversion used by the helper
library:

$$\text{PPD} = \tan(1^\circ) \cdot D_{\text{cm}} \cdot \frac{W_{\text{px}}}{W_{\text{cm}}}$$

where $W_{\text{px}}$ = screen width in pixels, $W_{\text{cm}}$ = monitor
physical width in cm, and $D_{\text{cm}}$ = viewing distance in cm.

`np.tan(np.radians(1.0))` gives the physical size of 1° at unit distance.


In [8]:
def pixels_per_degree_fn(
    screen_w_px: int,
    monitor_w_cm: float,
    viewing_dist_cm: float,
) -> float:
    """Compute pixels per degree of visual angle."""
    return float(np.tan(np.radians(1.0)) * viewing_dist_cm * (screen_w_px / monitor_w_cm))


In [9]:
def test_pixels_per_degree_fn(fn):
    ppd_lib = pixels_per_degree(3840, 71.0, 61.0)
    ppd_stu = fn(3840, 71.0, 61.0)
    assert isinstance(ppd_stu, float) and ppd_stu > 0
    assert np.isclose(ppd_stu, ppd_lib, rtol=1e-4), \
        f"Expected ≈ {ppd_lib:.3f}, got {ppd_stu:.3f}."
    assert fn(3840, 71.0, 61.0) > fn(1920, 71.0, 61.0)
    print(f"Exercise 9 passed.  PPD = {ppd_stu:.2f} px/°")


test_pixels_per_degree_fn(pixels_per_degree_fn)

Exercise 10 passed.  PPD = 57.59 px/°


## Pre-solved — Bivariate Contour Ellipse Area (BCEA)

**Precision** describes how *consistent* gaze is, independent of whether it
is accurate. The standard precision metric in eye-tracking is the **BCEA**
(Crossland & Rubin 2002), the area of the ellipse that encloses a fraction P
of the bivariate gaze distribution:

$$\text{BCEA} = 2\pi \, k \, \sigma_x \, \sigma_y \sqrt{1 - \rho^2}$$

where $k = -\ln(1 - P)$, $\sigma_x$ and $\sigma_y$ are the per-axis standard
deviations (ddof=1), and $\rho$ is the Pearson correlation between x and y.

The factor $\sqrt{1 - \rho^2}$ shrinks the area when x and y are correlated
(the ellipse is thin and tilted, so it covers less total area).


In [10]:
def bcea_area(
    xs: np.ndarray,
    ys: np.ndarray,
    p: float = 0.68,
) -> float:
    """Bivariate Contour Ellipse Area enclosing fraction p of the distribution."""
    import math
    if len(xs) < 3:
        return 0.0
    sx = float(np.std(xs, ddof=1))
    sy = float(np.std(ys, ddof=1))
    if sx == 0 or sy == 0:
        return 0.0
    rho = float(np.corrcoef(xs, ys)[0, 1])
    rho = float(np.clip(rho, -0.9999, 0.9999))
    k = -math.log(1.0 - p)
    return float(2.0 * math.pi * k * sx * sy * math.sqrt(1.0 - rho * rho))

In [11]:
def test_bcea_area(fn):
    rng = np.random.default_rng(42)
    xs = rng.normal(0, 10, 200)
    ys = rng.normal(0, 10, 200)

    val     = fn(xs, ys, p=0.68)
    val_lib = bcea(xs, ys, p=0.68)
    assert isinstance(val, float) and val > 0
    assert np.isclose(val, val_lib, rtol=1e-4)
    assert fn(xs, ys, 0.95) > fn(xs, ys, 0.68)
    assert fn(np.array([0.0, 1.0]), np.array([0.0, 1.0]), 0.68) == 0.0

    print("Pre-solved BCEA helper passed.")


test_bcea_area(bcea_area)

Pre-solved BCEA helper passed.


---

## Configuration

In [12]:
EXPERIMENT             = "IMRFSpatialAV"
PARTICIPANT_ID         = DEFAULT_PARTICIPANT_ID  # options: "p0096", "p0097", "p0099"
PREFERRED_RECORDING_ID = None        # optional Neon folder UUID override

SCREEN_W              = 1920
SCREEN_H              = 1080
MONITOR_WIDTH_CM      = 52.0
VIEWING_DISTANCE_CM   = 60.0
AOI_BUFFER_PCT        = 15.0
MIN_FIXATIONS_FOR_ELLIPSE = 3

TARGET_NAMES = {"Left", "Right", "Fixation Cross"}
AOI_METHOD   = "lrvt"   # options: "circle", "voronoi", "lrvt", "grid"
GRID_SIZE_PX = 200      # only used when AOI_METHOD = "grid"

WORKSHOP_ROOT = PROJECT_ROOT / "notebooks" / "workshop_output" / EXPERIMENT
NEON_ROOT = PROJECT_ROOT / "data_output" / EXPERIMENT / "neon"


def _latest_output_folder(root: Path) -> Path | None:
    if not root.is_dir():
        return None
    folders = [p for p in root.iterdir() if p.is_dir()]
    return max(folders, key=lambda p: p.stat().st_mtime) if folders else None


selected_recording = find_neon_recording(
    NEON_ROOT,
    participant_id=PARTICIPANT_ID,
    recording_id=PREFERRED_RECORDING_ID,
)
selected_recording_id = selected_recording.name if selected_recording else PREFERRED_RECORDING_ID
SELECTED_PARTICIPANT = neon_participant_id(selected_recording) if selected_recording else PARTICIPANT_ID

RECORDING_OUTPUT = (
    WORKSHOP_ROOT / selected_recording_id
    if selected_recording_id
    else _latest_output_folder(WORKSHOP_ROOT)
)

if RECORDING_OUTPUT is None:
    RECORDING_ID = "missing_recording"
    FIXATION_CSV = Path("missing_apriltag_mapped_fixations.csv")
else:
    RECORDING_ID = RECORDING_OUTPUT.name
    FIXATION_CSV = RECORDING_OUTPUT / "apriltag_mapped_fixations.csv"

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "reports" / "workshop_aoi" / RECORDING_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if FIXATION_CSV.is_file():
    fix_df = pd.read_csv(FIXATION_CSV)
else:
    fix_df = pd.DataFrame()
    print(f"Missing fixation export: {FIXATION_CSV}")
    print(f"Participant: {SELECTED_PARTICIPANT}")
print(f"Recording : {RECORDING_ID}")
print(f"Fixations : {len(fix_df):,}")
if not fix_df.empty:
    display(fix_df.head())


Recording : fdd1a874-8df3-4c46-b994-51272f98ec13
Fixations : 263


,start_time,stop_time,start_gaze_x,start_gaze_y,stop_gaze_x,stop_gaze_y,mean_gaze_x,mean_gaze_y,screen_x,screen_y,on_screen,frame_idx,event_x,event_y
0,1782119128061641499,1782119128407010499,903.19763,526.9613,897.40674,527.71313,898.57400,533.26310,1100.711850,587.969106,True,0,898.57400,533.26310
1,1782119128492009499,1782119128602132499,783.07620,910.1115,793.64105,905.03076,785.50960,900.45420,829.492352,1472.581198,False,8,785.50960,900.45420
2,1782119128667256499,1782119128917377499,711.11180,1177.1487,713.79290,1145.33750,717.92896,1150.53930,582.966844,2217.225601,False,13,717.92896,1150.53930
3,1782119129127624499,1782119129327870499,851.91394,538.2930,809.78754,519.58795,826.59576,522.40063,877.076320,694.449193,True,27,826.59576,522.40063
4,1782119129367870499,1782119129577992499,831.14594,623.9979,808.65857,624.62085,815.72955,625.07513,890.572446,900.521134,True,34,815.72955,625.07513


## Build AOIs and Classify Fixations

In [13]:
aois = build_paradigm_aois(
    screen_w_px=SCREEN_W,
    screen_h_px=SCREEN_H,
    monitor_w_cm=MONITOR_WIDTH_CM,
    viewing_dist_cm=VIEWING_DISTANCE_CM,
    buffer_pct=AOI_BUFFER_PCT,
)
aois = [a for a in aois if a.name in TARGET_NAMES]
ppd  = pixels_per_degree(SCREEN_W, MONITOR_WIDTH_CM, VIEWING_DISTANCE_CM)

display(pd.DataFrame([
    {"Target": a.name, "center_x": a.center_x, "center_y": a.center_y,
     "radius_px": a.radius, "radius_deg": a.radius / ppd}
    for a in aois
]))

,Target,center_x,center_y,radius_px,radius_deg
0,Left,785.98643,636.674206,66.705202,1.7250
1,Right,1134.01357,636.674206,66.705202,1.7250
2,Fixation Cross,960.00000,926.696823,33.352601,0.8625


In [14]:
if fix_df.empty:
    fix = pd.DataFrame()
    print("No fixations loaded.")
else:
    on_screen = fix_df.get("on_screen", pd.Series(True, index=fix_df.index)).astype(bool)
    fix = fix_df[on_screen].copy()
    fix = fix[np.isfinite(fix["screen_x"]) & np.isfinite(fix["screen_y"])].reset_index(drop=True)
    fix = classify_fixations_extended(fix, aois, ppd=ppd, method=AOI_METHOD)
    fix = fix.rename(columns={"is_hit": "inside_nearest_aoi"})
    print(f"On-screen valid fixations : {len(fix):,}")
    display(fix[["screen_x", "screen_y", "nearest_target",
                 "distance_to_nearest_deg", "inside_nearest_aoi"]].head())

On-screen valid fixations : 252


,screen_x,screen_y,nearest_target,distance_to_nearest_deg,inside_nearest_aoi
0,1100.711850,587.969106,Right,1.525785,True
1,877.076320,694.449193,Left,2.789450,True
2,890.572446,900.521134,Fixation Cross,1.918766,True
3,890.611929,798.309132,Fixation Cross,3.773982,True
4,1118.157709,539.345845,Right,2.550097,True


## Dispersion Metrics — Exercises Applied

The metric functions from Exercises 8 and 9 plus the pre-solved helpers are now applied
to the real fixation data for each target. The results are then compared
with the library functions `bcea` and `confidence_ellipse_params`.


In [15]:
metrics = []
ellipses = {}

if fix.empty:
    metrics_df = pd.DataFrame()
    print("No fixation data.")
else:
    ppd_ex9 = pixels_per_degree_fn(SCREEN_W, MONITOR_WIDTH_CM, VIEWING_DISTANCE_CM)

    for aoi in aois:
        subset = fix[fix["nearest_target"] == aoi.name]
        xs = subset["screen_x"].to_numpy(dtype=float)
        ys = subset["screen_y"].to_numpy(dtype=float)
        n  = len(subset)

        if n:
            bias_px       = euclidean_bias(xs, ys, aoi.center_x, aoi.center_y)
            rmse_px       = rmse_from_center(xs, ys, aoi.center_x, aoi.center_y)
            # Strict circular AOI hit count, independent of AOI_METHOD.
            n_in, frac_in = count_inside_aoi(xs, ys, aoi.center_x, aoi.center_y, aoi.radius)
            # Current AOI_METHOD hit count from classify_fixations_extended.
            n_method      = int(subset["inside_nearest_aoi"].astype(bool).sum()) if "inside_nearest_aoi" in subset else 0
            frac_method   = n_method / n
            bcea95        = bcea_area(xs, ys, p=0.95) if n >= MIN_FIXATIONS_FOR_ELLIPSE else np.nan
            sd_x          = float(np.std(xs, ddof=1)) if n > 1 else 0.0
            sd_y          = float(np.std(ys, ddof=1)) if n > 1 else 0.0
            ell           = confidence_ellipse_params(xs, ys, confidence=0.95) \
                            if n >= MIN_FIXATIONS_FOR_ELLIPSE else None
        else:
            bias_px = rmse_px = bcea95 = sd_x = sd_y = np.nan
            n_in, frac_in = 0, np.nan
            n_method, frac_method = 0, np.nan
            ell = None

        ellipses[aoi.name] = ell
        metrics.append({
            "Target": aoi.name,
            "N": n,
            "N_inside_circle": n_in,
            "Circle_AOI_pct": 100.0 * frac_in if np.isfinite(frac_in) else np.nan,
            "N_inside_method": n_method,
            "Method_AOI_pct": 100.0 * frac_method if np.isfinite(frac_method) else np.nan,
            "Bias_deg":   bias_px  / ppd_ex9 if np.isfinite(bias_px)  else np.nan,
            "RMSE_deg":   rmse_px  / ppd_ex9 if np.isfinite(rmse_px)  else np.nan,
            "SD_x_deg":  sd_x / ppd_ex9,
            "SD_y_deg":  sd_y / ppd_ex9,
            "BCEA95_deg2": bcea95 / (ppd_ex9 ** 2) if np.isfinite(bcea95) else np.nan,
        })

    metrics_df = pd.DataFrame(metrics)
    display(metrics_df.set_index("Target"))

,N,N_inside_AOI,Inside_AOI_pct,Bias_deg,RMSE_deg,SD_x_deg,SD_y_deg,BCEA95_deg2
Target,,,,,,,,
Left,87,13,14.942529,0.404313,3.924246,3.319034,2.097002,126.418623
Right,61,21,34.426230,1.079087,3.558913,2.533449,2.296686,104.397083
Fixation Cross,104,6,5.769231,1.988060,2.919778,1.625465,1.405340,41.676452


## Figure and Export

In [16]:
fig, ax = plt.subplots(figsize=(12, 7), constrained_layout=True)

if AOI_METHOD == "voronoi":
    regions = compute_voronoi_regions(aois, SCREEN_W, SCREEN_H)
    for aoi in aois:
        region = regions.get(aoi.name)
        if region is not None:
            color = AOI_COLORS.get(aoi.name, PLOT_COLORS["raw"])
            ax.add_patch(MplPolygon(region, closed=True, facecolor=color, alpha=0.10,
                                    edgecolor=color, linewidth=1.5, linestyle="--"))
elif AOI_METHOD == "grid":
    cells = compute_grid_cells(aois, grid_size_px=GRID_SIZE_PX,
                                screen_w_px=SCREEN_W, screen_h_px=SCREEN_H)
    for aoi in aois:
        color = AOI_COLORS.get(aoi.name, PLOT_COLORS["raw"])
        for (x, y, w, h) in cells.get(aoi.name, []):
            ax.add_patch(mpatches.Rectangle(
                (x, y), w, h, facecolor=color, alpha=0.12,
                edgecolor=PLOT_COLORS["arrow"], linewidth=0.4, zorder=1,
            ))

for aoi in aois:
    color = AOI_COLORS.get(aoi.name, PLOT_COLORS["raw"])
    if not fix.empty:
        subset = fix[fix["nearest_target"] == aoi.name]
        if not subset.empty:
            ax.scatter(subset["screen_x"], subset["screen_y"], s=14, alpha=0.55,
                       color=color, label=f"{aoi.name} (n={len(subset)})", zorder=4)
        ell = ellipses.get(aoi.name)
        if ell is not None:
            ax.add_patch(mpatches.Ellipse(
                (ell.center_x, ell.center_y),
                width=2 * ell.semi_major, height=2 * ell.semi_minor,
                angle=ell.angle_deg, fill=False, edgecolor=color,
                linestyle="--", linewidth=1.5, zorder=5,
            ))
    ax.add_patch(mpatches.Circle(
        (aoi.center_x, aoi.center_y), aoi.radius,
        fill=False, edgecolor=color, linewidth=1.8, zorder=3,
    ))
    ax.scatter([aoi.center_x], [aoi.center_y], marker="+", s=120,
               color=color, linewidths=2, zorder=3)
    ax.text(aoi.center_x, aoi.center_y + aoi.radius + 30, aoi.name,
            color=color, ha="center", fontsize=9)

if fix.empty:
    ax.text(SCREEN_W / 2, SCREEN_H / 2,
            "No fixations loaded.\nRun the surface mapping workshop first.",
            ha="center", va="center", fontsize=13,
            bbox=dict(boxstyle="round,pad=0.5",
                      facecolor=PLOT_COLORS["box_face"],
                      edgecolor=PLOT_COLORS["off_screen"]))

ax.set_xlim(-50, SCREEN_W + 50)
ax.set_ylim(SCREEN_H + 50, -50)
ax.set_aspect("equal")
ax.set_xlabel("Screen X (px)")
ax.set_ylabel("Screen Y (px)")
ax.set_title(f"AOI Dispersion [{AOI_METHOD}] — {RECORDING_ID}")
if not fix.empty:
    ax.legend(loc="upper right", fontsize=8, framealpha=0.9)

fig_path = OUTPUT_DIR / f"aoi_dispersion_{RECORDING_ID}.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved figure -> {fig_path}")

if not fix.empty:
    metrics_df.to_csv(OUTPUT_DIR / f"aoi_dispersion_metrics_{RECORDING_ID}.csv", index=False)
    fix.to_csv(OUTPUT_DIR / f"aoi_classified_fixations_{RECORDING_ID}.csv", index=False)
    print(f"Saved metrics + classified fixations -> {OUTPUT_DIR}")

Saved figure -> /Users/eduardo/Workspace/Unity/UnityProjects/IMRF_2026_SpatialAudio_Eyetracking/IMRF_Eyetracking/notebooks/reports/workshop_aoi/fdd1a874-8df3-4c46-b994-51272f98ec13/aoi_dispersion_fdd1a874-8df3-4c46-b994-51272f98ec13.png
Saved metrics + classified fixations -> /Users/eduardo/Workspace/Unity/UnityProjects/IMRF_2026_SpatialAudio_Eyetracking/IMRF_Eyetracking/notebooks/reports/workshop_aoi/fdd1a874-8df3-4c46-b994-51272f98ec13


/var/folders/n5/rzkmrrl9699cvlrds_01_ww00000gn/T/ipykernel_50599/2768593596.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Short Reflection

Discuss or answer the following:

1. **Bias vs RMSE.** Looking at the table, which target has higher RMSE
   relative to its bias? What does this tell you about the shape of the
   fixation scatter for that target?
2. **BCEA interpretation.** The BCEA at p=0.95 encloses 95 % of
   the bivariate scatter. How would you expect the BCEA to change if you
   increased the lowpass filter cutoff in the surface mapping notebook?
   Why?
3. **Method comparison.** Re-run with `AOI_METHOD = "voronoi"`. Compare
   `Circle_AOI_pct` with `Method_AOI_pct`. Which column changes, and why
   does Voronoi classify fixations differently from a strict circular AOI?
4. **Pixels per degree.** At the notebook setup (1920 px wide, 52 cm
   screen, 60 cm viewing distance), how many pixels correspond to 1 °?
   At 2 °? Is the AOI radius (≈ 1.7 °) reasonable for this setup?
5. **Small-angle approximation.** If you replace `tan(1°)` with 1° in
   radians, by how many pixels per degree does the approximation differ?
   Why is this difference small here?
